In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()


True

## Encode the image and send to the vision model

In [13]:
with open("blood_report.png","rb") as f:
    image_64 = base64.b64encode(f.read()).decode()
image_64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAWUAAAG1CAIAAACu2XLDAAAAA3NCSVQICAjb4U/gAAAa/3pUWHRSYXcgcHJvZmlsZSB0eXBlIEFQUDEAAGiBtZpZklw5ckX/axW9BMDhmJaDUdZmMknW+//QuYggi6wu05+YVczMYMR7gOP6HYD3x3+c/zr/+uf6x//867/vP//z/PEP'

### The above converting image to string

In [3]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
)

In [5]:
message = HumanMessage(content=[
    {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_64}"}},
    {"type":"text","text":"This is a blood work report. Extract all the test results and flag any values outside the normal range"}
])
response = llm.invoke([message])
print(response.content)

Based on the blood work report provided, here is the extraction of all test results and any values outside the normal range:

### Patient Information
- **Patient:** Rajesh Sharma
- **Age:** 48
- **Gender:** Male
- **Date:** May 7, 2026

### Complete Blood Count (CBC)
- **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) - Within normal range
- **Hematocrit:** 44% (Normal: 41-53%) - Within normal range
- **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0) - Within normal range
- **Platelets:** 220 x10^3/uL (Normal: 150-400) - Within normal range

### Lipid Panel
- **Total Cholesterol:** 238 mg/dL (Normal: <200) - **Outside normal range**
- **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **Outside normal range**
- **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Outside normal range**
- **Triglycerides:** 188 mg/dL (Normal: <150) - **Outside normal range**

### Metabolic Panel
- **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - Within normal range
- **HbA1c:** 5.3% (Normal: <5.7%) - Within normal range
- 

In [9]:
from langchain_core.tools import tool
@tool
def get_diet_recommendation(condition: str) -> dict:
  """Given a health condition, returns a diet plan.

  Condition must be one of: normal, high_cholesterol, high_sugar
  """
  diet_plans = {
      "high_cholesterol": {
          "eat": ["fruits", "vegetables", "whole grains", "lean protein"],
          "do_not_eat": [
              "red meat",
              "fried food",
              "full-fat dairy",
              "processed snacks",
          ],
      },
      "high_sugar": {
          "eat": ["vegetables", "whole grains", "legumes", "nuts"],
          "do_not_eat": [
              "white rice",
              "white sugar",
              "junk food",
              "sugary drinks",
          ],
      },
      "normal": {
          "eat": ["vegetables", "fruits", "whole grains", "lean protein"],
          "do_not_eat": [
              "excessive sugar",
              "processed food",
              "trans fats",
          ],
      },
  }
  return diet_plans.get(condition, diet_plans["normal"])


In [11]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [15]:
result = diet_agent.invoke({
    "messages": [
        HumanMessage(content=[
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_64}"},
            },
            {
                "type": "text",
                "text": "Analyse this blood work report and suggest a diet plan.",
            },
        ])
    ]
})

print(result["messages"][-1].content)

Based on the blood work report provided for Rajesh Sharma, here is the analysis:

**1. Complete Blood Count (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5–17.5) — **Normal**
*   **Hematocrit:** 44% (Normal: 41–53%) — **Normal**
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5–11.0) — **Normal**
*   **Platelets:** 220 x10^3/uL (Normal: 150–400) — **Normal**

**2. Lipid Panel**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) — **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) — **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) — **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) — **High**

**3. Metabolic Panel**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70–99) — **Normal**
*   **HbA1c:** 5.3% (Normal: <5.7%) — **Normal**
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7–1.3) — **Normal**
*   **eGFR:** 82 mL/min (Normal: >60) — **Normal**

**4. Liver Function**
*   **ALT:** 28 U/L (Normal: 7–40) — **Normal**
*   **AST:** 25 U/L (Normal: 10–40) — **Normal**
*   **Bilirubin